<div align="center">

# Notebook 6

# Classification multi-class FOCH

**Transfer Learning VoxCeleb — Branche Dual pour la classification**

![Python](https://img.shields.io/badge/Python-3.10+-blue?style=flat-square&logo=python)
![PyTorch](https://img.shields.io/badge/PyTorch-2.0+-EE4C2C?style=flat-square&logo=pytorch)
![HuggingFace](https://img.shields.io/badge/🤗-Transformers-FFD21F?style=flat-square)
![CUDA](https://img.shields.io/badge/CUDA-AMP%20Enabled-76B900?style=flat-square&logo=nvidia)

</div>


## 1. Configuration de l'environnement et des chemins

Cette première section centralise les dépendances et les chemins. Le notebook met en œuvre un **modèle de fusion bi-branche** (WavLM-Large + ResNetSE34L VoxCeleb) évalué par **validation croisée stratifiée à 5 plis** (`StratifiedKFold`), avec **augmentation de données injectée pli par pli dans le seul jeu d'entraînement**.

**Classes retenues :**

| Classe | Origine | Justification |
|---|---|---|
| `PR` | FOCH | Pathologie majoritaire (support le plus élevé) |
| `LMB` | FOCH | Support intermédiaire |
| `fuite glottique` | FOCH | Support intermédiaire |
| `Healthy` | Saarbruecken (SVD) | Groupe contrôle — voyelle /a/ à hauteur habituelle |

La classe `Cancer` reste **exclue** (support insuffisant, n ≈ 26). Le déséquilibre des trois pathologies est compensé par des **signaux augmentés** (répertoire `Segmented_Voyelles_Augmented/signal`), injectés **exclusivement dans le train de chaque pli** ; le pli de validation reste strictement original.

> 💡 **Architecture (Transfer Learning VoxCeleb)** — Branche A = **WavLM-Large** local (embeddings gelés, 1024-d, microscopique) ; Branche B = **ResNetSE34L** (pré-entraîné sur VoxCeleb pour la reconnaissance du locuteur, 512-d, voix-spécifique) → fusion **1536-d**. ResNetSE34L prend la forme d'onde brute 16 kHz et calcule son propre spectrogramme de Mel (n\_mels=40) en interne. Poids et métriques sont sauvegardés dans `.poids_modèles` et `.foch_métriques`.


In [7]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 1 — Imports, configuration des chemins et hyperparamètres
# ═══════════════════════════════════════════════════════════════════════════
import os
import gc
import glob
import random
import sys
import warnings

import numpy as np
import pandas as pd
import torch
import torchaudio
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')

# ── Reproductibilité ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── FFmpeg (backend de décodage audio pour torchaudio) ───────────────────────
import shutil
_ffmpeg_bin = r'C:/Users/AdminIA/Downloads/ffmpeg-8.1.1-full_build-shared/ffmpeg-8.1.1-full_build-shared/bin'
if os.path.isdir(_ffmpeg_bin):
    os.environ['PATH'] += os.pathsep + _ffmpeg_bin
print(f'FFmpeg : {shutil.which("ffmpeg")}')

# ── Chemin du trainer VoxCeleb (ajouté au sys.path pour importer ResNetSE34L) ─
VOXCELEB_TRAINER_DIR = r'C:/Users/AdminIA/Desktop/Vox_celeb/voxceleb_trainer'
if VOXCELEB_TRAINER_DIR not in sys.path:
    sys.path.insert(0, VOXCELEB_TRAINER_DIR)

# ── Chemins des données ──────────────────────────────────────────────────────
CSV_PATH        = r'C:/Users/AdminIA/Desktop/ORL_IA_FOCH_Callum_HOLLIDAY/csv_best/orl_df_vowel_master.csv'
FOCH_AUDIO_DIR  = r'C:/Users/AdminIA/Desktop/ORL_IA_FOCH_Callum_HOLLIDAY/Segmented_Voyelles_Best'
SVD_HEALTHY_DIR = r'C:/Users/AdminIA/Documents/SVD/healthy'

# ── Données augmentées ────────────────────────────────────────────────────────
AUG_DIR         = r'C:/Users/AdminIA/Desktop/ORL_IA_FOCH_Callum_HOLLIDAY/Segmented_Voyelles_Augmented'
AUG_SIGNAL_DIR  = os.path.join(AUG_DIR, 'signal')
AUG_MANIFESTS   = [
    os.path.join(AUG_DIR, 'manifest_train_bal.csv'),
    os.path.join(AUG_DIR, 'manifest_train_bal_vc.csv'),
]

# ── Backbones et artefacts de sortie ─────────────────────────────────────────
MODEL_DIR   = r'C:/Users/AdminIA/Documents/models/wavlm-large'
WEIGHTS_DIR = r'C:/Users/AdminIA/Documents/models/.poids_modèles'
METRICS_DIR = r'C:/Users/AdminIA/Documents/models/.foch_métriques'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)
RUN_NAME = 'wavlm_voxceleb_kfold_aug'

# ── Poids VoxCeleb pré-entraînés — stratégie de chargement ──────────────────
# Téléchargeable depuis : http://www.robots.ox.ac.uk/~joon/data/baseline_lite_ap.model
# Ou utiliser un modèle entraîné localement avec voxceleb_trainer.
VOXCELEB_CKPT_PATH  = r'C:/Users/AdminIA/Desktop/Vox_celeb/voxceleb_trainer/baseline_lite_ap.model'
VOXCELEB_PRETRAINED = True   # mettre False pour ignorer tout chargement de poids
VOXCELEB_EMB_DIM    = 512    # nOut du réseau ResNetSE34L (SAP) — baseline_lite_ap.model utilise 512

# ── Définition des classes ───────────────────────────────────────────────────
TARGET_PATHOLOGIES = ['PR', 'LMB', 'fuite glottique']
HEALTHY_LABEL      = 'Healthy'
CLASSES            = TARGET_PATHOLOGIES + [HEALTHY_LABEL]
NUM_LABELS         = len(CLASSES)
N_HEALTHY          = 100

# ── Validation croisée ────────────────────────────────────────────────────────
K_FOLDS      = 5

# ── Hyperparamètres audio ─────────────────────────────────────────────────────
TARGET_SR    = 16000
MAX_LEN_S    = 4.0
MAX_LEN      = int(TARGET_SR * MAX_LEN_S)

# ── Hyperparamètres d'entraînement ────────────────────────────────────────────
BATCH_SIZE      = 4
ACCUM_STEPS     = 4
EPOCHS          = 30
LR              = 1e-4
WEIGHT_DECAY    = 0.01
PATIENCE        = 6
DROPOUT         = 0.3
HIDDEN_DIM      = 512
FREEZE_WAVLM    = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Résumé de la configuration VoxCeleb ─────────────────────────────────────
if not VOXCELEB_PRETRAINED:
    _vc_status = 'initialisation ALÉATOIRE (VOXCELEB_PRETRAINED=False)'
elif VOXCELEB_CKPT_PATH and os.path.isfile(VOXCELEB_CKPT_PATH):
    _vc_status = f'chemin LOCAL → {VOXCELEB_CKPT_PATH}'
else:
    _vc_status = f'FICHIER INTROUVABLE — {VOXCELEB_CKPT_PATH} → initialisation aléatoire'

print('Configuration chargée.')
print(f'  Classes ({NUM_LABELS}) : {CLASSES}')
print(f'  Healthy cible   : {N_HEALTHY} (sous-échantillonné)')
print(f'  Validation      : StratifiedKFold à {K_FOLDS} plis')
print(f'  Backbone WavLM  : {MODEL_DIR}')
print(f'  Fusion          : 1024 (WavLM) + {VOXCELEB_EMB_DIM} (VoxCeleb ResNetSE34L) = {1024 + VOXCELEB_EMB_DIM}')
print(f'  VoxCeleb poids  : {_vc_status}')
print(f'  Device          : {device}')


FFmpeg : C:/Users/AdminIA/Downloads/ffmpeg-8.1.1-full_build-shared/ffmpeg-8.1.1-full_build-shared/bin\ffmpeg.EXE
Configuration chargée.
  Classes (4) : ['PR', 'LMB', 'fuite glottique', 'Healthy']
  Healthy cible   : 100 (sous-échantillonné)
  Validation      : StratifiedKFold à 5 plis
  Backbone WavLM  : C:/Users/AdminIA/Documents/models/wavlm-large
  Fusion          : 1024 (WavLM) + 512 (VoxCeleb ResNetSE34L) = 1536
  VoxCeleb poids  : chemin LOCAL → C:/Users/AdminIA/Desktop/Vox_celeb/voxceleb_trainer/baseline_lite_ap.model
  Device          : cuda


## 2. Parsing des métadonnées, nettoyage et appariement Saarbruecken

Cette étape transforme les métadonnées brutes (`orl_df_vowel_master.csv`) en un jeu de données exploitable. Elle enchaîne quatre opérations atomiques :

1. **Normalisation défensive** des chaînes de caractères — les libellés `VSL` contiennent du bruit (espaces parasites, casse hétérogène, valeurs vides), source classique d'erreurs d'appariement *silencieuses*.
2. **Filtrage** sur les trois pathologies cibles, avec réattribution d'un libellé canonique.
3. **Déduplication patient** sur la clé `Last_Name` : un même patient pouvant disposer de plusieurs enregistrements (pré/post-opératoire, suivi), on n'en conserve qu'un afin de prévenir toute **fuite d'information** entre les jeux d'entraînement et de test.
4. **Appariement** ligne CSV ↔ fichier audio, avec vérification de l'existence physique de chaque fichier sur le disque.

> 🔎 **Constat de l'audit des données** — Le champ `File_Name` correspond *exactement* aux fichiers présents dans `Segmented_Voyelles_Best` (281 fichiers, 329 lignes, appariement à 100 %). Aucun appariement flou n'est donc nécessaire côté FOCH : la fragilité réelle réside dans le **bruit des libellés** (`'Cancer '` avec espace, valeurs vides, `'?'`) et les **doublons de lignes**.

> ⚠️ **Limite connue** — `Last_Name` est un identifiant *proxy* imparfait : des homonymes (p. ex. `MARTIN`, présent 7 fois) peuvent être fusionnés à tort lors de la déduplication. Ce point devra être audité via `patient_id`.

Le **groupe contrôle sain** est extrait de la base **Saarbruecken (SVD)** selon la logique établie dans `1_0_foch_test.ipynb` : seuls les fichiers de voyelle /a/ à hauteur habituelle (suffixe `a_h`) sont retenus.

In [8]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 2 — Parsing des métadonnées, nettoyage et appariement Saarbruecken
# ═══════════════════════════════════════════════════════════════════════════

# ── 1. Lecture des métadonnées FOCH ──────────────────────────────────────────
df_meta = pd.read_csv(CSV_PATH)
print(f'Métadonnées brutes            : {len(df_meta):>4} lignes')

# ── 2. Normalisation défensive des chaînes de caractères ─────────────────────
# Les libellés VSL comportent du bruit : espaces parasites ('Cancer ' vs
# 'Cancer'), valeurs vides/NaN, point d'interrogation. Ce nettoyage systématique
# évite les erreurs d'appariement silencieuses sur les libellés de pathologie.
for col in ['Last_Name', 'File_Name', 'VSL']:
    df_meta[col] = df_meta[col].astype(str).str.strip()

# Clé d'appariement insensible à la casse pour les libellés de pathologie
df_meta['VSL_norm'] = df_meta['VSL'].str.lower()

# ── 3. Filtrage sur les classes cibles du baseline ───────────────────────────
# Dictionnaire { libellé_normalisé : libellé_canonique } pour réattribuer
# proprement le nom de classe officiel après le filtrage.
target_map = {p.lower(): p for p in TARGET_PATHOLOGIES}
df_patho = df_meta[df_meta['VSL_norm'].isin(target_map)].copy()
df_patho['label_str'] = df_patho['VSL_norm'].map(target_map)
print(f'Lignes pathologiques retenues : {len(df_patho):>4}  (PR / LMB / fuite glottique)')

# ── 4. Déduplication au niveau patient (clé = Last_Name) ─────────────────────
# Un même patient peut posséder plusieurs enregistrements (pré/post-opératoire,
# suivi longitudinal). On ne conserve qu'un enregistrement par patient afin de
# prévenir toute fuite d'information entre les sous-ensembles train/test.
# NB : Last_Name est un proxy imparfait — des homonymes (ex. MARTIN) peuvent
#      être fusionnés à tort ; à auditer ultérieurement via patient_id.
n_avant = len(df_patho)
df_patho = df_patho.drop_duplicates(subset='Last_Name', keep='first')
print(f'Après déduplication patient    : {len(df_patho):>4}  ({n_avant - len(df_patho)} doublon(s) retiré(s))')

# ── 5. Appariement robuste : ligne CSV ↔ fichier audio sur disque ────────────
# Le champ File_Name correspond exactement au nom de fichier présent dans le
# répertoire FOCH. On vérifie néanmoins l'existence physique de chaque fichier
# (tolérance aux fautes) et l'on écarte proprement les éventuels manquants.
def resoudre_chemin_foch(nom_fichier):
    chemin = os.path.join(FOCH_AUDIO_DIR, nom_fichier)
    return chemin if os.path.isfile(chemin) else None

df_patho['filepath'] = df_patho['File_Name'].apply(resoudre_chemin_foch)
n_manquants = df_patho['filepath'].isna().sum()
if n_manquants:
    print(f'⚠  {n_manquants} fichier(s) FOCH introuvable(s) sur disque — exclus.')
df_patho = df_patho.dropna(subset=['filepath'])
df_patho = df_patho[['filepath', 'File_Name', 'Last_Name', 'label_str']]

# ── 6. Groupe contrôle sain — base Saarbruecken (SVD) ────────────────────────
# Logique reprise de 1_0_foch_test.ipynb : on ne retient que la voyelle /a/ à
# hauteur habituelle (suffixe 'a_h'). Le motif '*a_h.wav' (et non '*a_h*')
# exclut volontairement les variantes a_hl / a_hn (hauteurs basse / haute).
svd_ah_files = glob.glob(os.path.join(SVD_HEALTHY_DIR, '*', 'vowels', '*a_h.wav'))
print(f'\nEnregistrements sains SVD a_h disponibles : {len(svd_ah_files):>4}')

# Sous-échantillonnage du groupe contrôle à ~N_HEALTHY enregistrements afin
# d'obtenir un problème de classification équilibré et représentatif. Sans cela,
# une accuracy élevée refléterait surtout la classe majoritaire 'Healthy'.
random.seed(SEED)
svd_ah_files = random.sample(svd_ah_files, min(N_HEALTHY, len(svd_ah_files)))
print(f'Enregistrements sains retenus (sous-échantillon)  : {len(svd_ah_files):>4}')

df_healthy = pd.DataFrame({
    'filepath':  svd_ah_files,
    'File_Name': [os.path.basename(f) for f in svd_ah_files],
    # Identifiant patient SVD = nom du dossier <patient_id>/vowels/<fichier>
    'Last_Name': [os.path.basename(os.path.dirname(os.path.dirname(f))) for f in svd_ah_files],
    'label_str': HEALTHY_LABEL,
})

# ── 7. Consolidation du jeu de données baseline ──────────────────────────────
df = pd.concat([df_patho, df_healthy], ignore_index=True)

print('\n' + '=' * 60)
print('Répartition des classes (baseline équilibré) :')
print('=' * 60)
print(df['label_str'].value_counts().reindex(CLASSES).to_string())
print('-' * 60)
print(f'TOTAL : {len(df)} enregistrements | {df["label_str"].nunique()} classes')

Métadonnées brutes            :  329 lignes
Lignes pathologiques retenues :  217  (PR / LMB / fuite glottique)
Après déduplication patient    :  146  (71 doublon(s) retiré(s))

Enregistrements sains SVD a_h disponibles :  687
Enregistrements sains retenus (sous-échantillon)  :  100

Répartition des classes (baseline équilibré) :
label_str
PR                  72
LMB                 48
fuite glottique     26
Healthy            100
------------------------------------------------------------
TOTAL : 246 enregistrements | 4 classes


## 3. Encodage des labels et mappings du modèle

La tête de classification de HuggingFace raisonne sur des **indices entiers**. On établit donc un encodage déterministe `label_str → label` selon l'ordre canonique défini en Cellule 1 (`CLASSES`), ainsi que les dictionnaires `label2id` / `id2label` qui seront transmis au modèle pour garantir la traçabilité clinique des prédictions.

In [9]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 3 — Encodage des labels et mappings modèle
# ═══════════════════════════════════════════════════════════════════════════
# Mappings bidirectionnels exigés par la tête de classification HuggingFace.
label2id = {c: i for i, c in enumerate(CLASSES)}
id2label = {i: c for c, i in label2id.items()}

df['label'] = df['label_str'].map(label2id).astype(int)

print('Mapping label → id :')
for c, i in label2id.items():
    print(f'  {i} : {c}')

# Garde-fou : tout libellé non mappé (NaN) trahirait une incohérence de classe.
assert df['label'].notna().all(), 'Libellé non mappé détecté !'

print('\nEffectifs par classe encodée :')
print(df.groupby(['label', 'label_str']).size().to_string())

Mapping label → id :
  0 : PR
  1 : LMB
  2 : fuite glottique
  3 : Healthy

Effectifs par classe encodée :
label  label_str      
0      PR                  72
1      LMB                 48
2      fuite glottique     26
3      Healthy            100


## 4. Processor WavLM et `Dataset` bi-branche

Le `Dataset` encapsule la chaîne de prétraitement et produit, pour chaque enregistrement, les **deux entrées** attendues par le modèle de fusion :

1. **Chargement** du `.wav` via `torchaudio`, **conversion mono**, **ré-échantillonnage** à 16 kHz, puis **mise à longueur fixe** (`MAX_LEN`) par troncature / zéro-padding ;
2. **Branche A (WavLM)** — extraction de features via le `Wav2Vec2FeatureExtractor` (normalisation + padding) → `input_values` ;
3. **Branche B (VoxCeleb)** — forme d'onde brute `wave` (MAX_LEN échantillons, 16 kHz) passée directement à **ResNetSE34L**, qui calcule son propre spectrogramme de Mel (n_mels=40, sr=16000) en interne avec InstanceNorm. Aucun calcul de Mel côté Dataset.

> 🛡️ **Robustesse** — un fichier corrompu ne doit jamais interrompre l'entraînement : un bloc `try/except` renvoie des tenseurs nuls de secours en signalant le fichier fautif.

> ℹ️ WavLM réutilise l'extracteur de features de **Wav2Vec2** (`Wav2Vec2FeatureExtractor`) ; il n'existe pas de classe « WavLMFeatureExtractor » dédiée.


In [10]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 4 — Processor WavLM + Dataset bi-branche (WavLM + VoxCeleb)
# ═══════════════════════════════════════════════════════════════════════════
from transformers import Wav2Vec2FeatureExtractor

# WavLM réutilise l'extracteur de features de Wav2Vec2 (normalisation + padding).
processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_DIR)
print(f'✓ Processor chargé : {type(processor).__name__}')


class FochVowelDataset(Dataset):
    """Charge un enregistrement de voyelle et produit DEUX représentations pour
    le modèle de fusion bi-branche :

      • `input_values` — forme d'onde brute (16 kHz, normalisée) destinée à la
        Branche A (WavLM-Large), via le `Wav2Vec2FeatureExtractor` ;
      • `wave`         — forme d'onde brute 16 kHz (MAX_LEN échantillons), destinée
        à la Branche B (ResNetSE34L VoxCeleb) qui calcule son propre spectrogramme
        de Mel en interne (n_mels=40, sr=16000).

    La forme d'onde est ramenée mono + 16 kHz + longueur fixe `MAX_LEN` par
    troncature / zéro-padding. Renvoie le triplet (input_values, wave, label)."""

    def __init__(self, dataframe, processor, target_sr=TARGET_SR, max_len=MAX_LEN):
        self.df        = dataframe.reset_index(drop=True)
        self.processor = processor
        self.sr        = target_sr
        self.max_len   = max_len

    def __len__(self):
        return len(self.df)

    def _fixed_length(self, wave):
        """Tronque ou complète (zéro-padding) la forme d'onde à `max_len`."""
        n = wave.shape[-1]
        if n >= self.max_len:
            return wave[..., :self.max_len]
        return torch.nn.functional.pad(wave, (0, self.max_len - n))

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = int(row['label'])
        try:
            waveform, orig_sr = torchaudio.load(row['filepath'])
            if waveform.shape[0] > 1:                                   # stéréo → mono
                waveform = waveform.mean(dim=0, keepdim=True)
            if orig_sr != self.sr:                                      # ré-échantillonnage
                waveform = torchaudio.transforms.Resample(orig_sr, self.sr)(waveform)
            wave = self._fixed_length(waveform.squeeze(0))              # (MAX_LEN,)

            # ── Branche A : WavLM (normalisation + padding via le processor) ──
            out = self.processor(
                wave.numpy(),
                sampling_rate=self.sr,
                max_length=self.max_len,
                truncation=True,
                padding='max_length',
                return_tensors='pt',
            )
            input_values = out.input_values.squeeze(0)                  # (MAX_LEN,)

            # ── Branche B : forme d'onde brute pour ResNetSE34L VoxCeleb ──────────────
            # ResNetSE34L calcule son propre mel (n_mels=40, sr=16000) en interne.
        except Exception as e:
            # Garde-fou : un fichier corrompu ne doit pas interrompre le DataLoader.
            print(f'[ATTENTION] {row["filepath"]} : {e}')
            input_values = torch.zeros(self.max_len)
            wave         = torch.zeros(self.max_len)
        return input_values, wave, torch.tensor(label, dtype=torch.long)


print(f'Fenêtre audio : {MAX_LEN_S} s → {MAX_LEN} échantillons à {TARGET_SR} Hz')
print(f'Branche A (WavLM)    : input_values ({MAX_LEN},)')
print(f'Branche B (VoxCeleb) : wave ({MAX_LEN},) → mel interne n_mels=40')


✓ Processor chargé : Wav2Vec2FeatureExtractor
Fenêtre audio : 4.0 s → 64000 échantillons à 16000 Hz
Branche A (WavLM)    : input_values (64000,)
Branche B (VoxCeleb) : wave (64000,) → mel interne n_mels=40


## 5. Préparation de la validation croisée (`StratifiedKFold`) + pool augmenté

On remplace le découpage train/val unique par une **validation croisée stratifiée à K = 5 plis** (`StratifiedKFold`, `shuffle=True`, `random_state=SEED`). La stratification préserve la proportion de chaque classe dans chaque pli ; la déduplication patient (Cellule 2) garantit un unique enregistrement par patient FOCH, de sorte que le découpage par enregistrement équivaut à un découpage patient (pas de fuite train ↔ validation).

**Pool augmenté.** Les deux manifestes (`manifest_train_bal.csv`, `manifest_train_bal_vc.csv`) sont **fusionnés** et dédupliqués ; seuls les segments `is_aug=True` (existence vérifiée sur disque) constituent le `aug_df`. Ce pool n'est rattaché à aucun pli ici : il sera **injecté pli par pli** (Cellule 7) **uniquement dans le train**, après exclusion de tout segment issu d'un patient présent en validation.

> 💡 Un batch « sonde » est extrait en fin de cellule afin de valider les dimensions du modèle de fusion (Cellule 6).

In [11]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 5 — Préparation de la validation croisée (StratifiedKFold) + pool augmenté
# ═══════════════════════════════════════════════════════════════════════════
from sklearn.model_selection import StratifiedKFold

# ── 1. Plis de validation croisée stratifiée ─────────────────────────────────
# StratifiedKFold préserve la proportion de chaque classe dans chaque pli. La
# déduplication patient (Cellule 2) garantit un unique enregistrement par patient
# FOCH ; les sujets SVD sont distincts. Au niveau enregistrement, le découpage
# équivaut donc à un découpage patient (pas de fuite entre train et validation).
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(df, df['label']))
print(f'StratifiedKFold : {K_FOLDS} plis générés sur {len(df)} enregistrements originaux.')
for k, (tr, va) in enumerate(folds):
    dist = df.iloc[va]['label_str'].value_counts().reindex(CLASSES).to_dict()
    print(f'  Pli {k+1} : train={len(tr):>3} | val={len(va):>2} | val par classe={dist}')

# ── 2. Construction du pool de données augmentées (fusion des manifestes) ────
# Les deux manifestes sont concaténés puis dédupliqués sur `filepath`. Seules les
# lignes augmentées (`is_aug=True`) sont conservées : elles seront injectées,
# pli par pli (Cellule 7), UNIQUEMENT dans le train — jamais en validation.
man_list = []
for mpath in AUG_MANIFESTS:
    if os.path.isfile(mpath):
        man_list.append(pd.read_csv(mpath))
        print(f'  Manifeste chargé : {os.path.basename(mpath)} ({len(man_list[-1])} lignes)')
    else:
        print(f'  ⚠  Manifeste introuvable : {mpath}')

if man_list:
    man = pd.concat(man_list, ignore_index=True).drop_duplicates(subset='filepath')
    for col in ['Last_Name', 'File_Name', 'label_str']:
        man[col] = man[col].astype(str).str.strip()
    man['is_aug_bool'] = man['is_aug'].astype(str).str.strip().str.lower() == 'true'

    aug_df = man[man['is_aug_bool']].copy()
    aug_df['label'] = aug_df['label_str'].map(label2id)
    aug_df = aug_df.dropna(subset=['label'])
    aug_df['label']  = aug_df['label'].astype(int)
    aug_df['exists'] = aug_df['filepath'].apply(os.path.isfile)
    n_missing = int((~aug_df['exists']).sum())
    if n_missing:
        print(f'  → {n_missing} fichier(s) augmenté(s) introuvable(s) — écarté(s)')
    aug_df = aug_df[aug_df['exists']]
    aug_df = aug_df[['filepath', 'File_Name', 'Last_Name', 'label_str', 'label']].reset_index(drop=True)
else:
    aug_df = pd.DataFrame(columns=['filepath', 'File_Name', 'Last_Name', 'label_str', 'label'])

print(f'\nPool augmenté disponible : {len(aug_df)} segments')
print(aug_df['label_str'].value_counts().reindex(CLASSES).fillna(0).astype(int).to_string())

# ── 3. Batch sonde (vérification dimensionnelle pour la Cellule 6) ───────────
# Un mini-DataLoader sur les premiers enregistrements fournit un batch type
# (deux entrées + labels) pour valider les dimensions du modèle de fusion.
_probe_loader = DataLoader(FochVowelDataset(df.head(BATCH_SIZE), processor),
                           batch_size=BATCH_SIZE, shuffle=False)
xb, wb, yb = next(iter(_probe_loader))
print(f'\nBatch sonde — WavLM : {tuple(xb.shape)} | VoxCeleb wave : {tuple(wb.shape)} | labels : {yb.tolist()}')

StratifiedKFold : 5 plis générés sur 246 enregistrements originaux.
  Pli 1 : train=196 | val=50 | val par classe={'PR': 15, 'LMB': 10, 'fuite glottique': 5, 'Healthy': 20}
  Pli 2 : train=197 | val=49 | val par classe={'PR': 15, 'LMB': 9, 'fuite glottique': 5, 'Healthy': 20}
  Pli 3 : train=197 | val=49 | val par classe={'PR': 14, 'LMB': 9, 'fuite glottique': 6, 'Healthy': 20}
  Pli 4 : train=197 | val=49 | val par classe={'PR': 14, 'LMB': 10, 'fuite glottique': 5, 'Healthy': 20}
  Pli 5 : train=197 | val=49 | val par classe={'PR': 14, 'LMB': 10, 'fuite glottique': 5, 'Healthy': 20}
  Manifeste chargé : manifest_train_bal.csv (320 lignes)
  Manifeste chargé : manifest_train_bal_vc.csv (320 lignes)

Pool augmenté disponible : 123 segments
label_str
PR                 22
LMB                42
fuite glottique    59
Healthy             0

Batch sonde — WavLM : (4, 64000) | VoxCeleb wave : (4, 64000) | labels : [0, 0, 2, 0]


## 6. Modèle de fusion bi-branche (WavLM-Large + VoxCeleb ResNetSE34L)

Réseau **bi-branche** dont les représentations globales sont concaténées avant la tête de classification. Une fonction `build_model()` permet de réinitialiser une instance neuve à chaque pli de la validation croisée.

### Branche A — Acoustique microscopique (`WavLM-Large`)
Backbone **WavLM-Large** local, **gelé** (extraction d'embeddings). Moyenne temporelle des états cachés → vecteur **1024-d**. Cible : jitter, shimmer, apériodicité court-terme.

### Branche B — Voice embeddings (`ResNetSE34L` — VoxCeleb)
**ResNetSE34L** pré-entraîné sur **VoxCeleb** (reconnaissance du locuteur à partir de centaines de milliers d'utterances de milliers de locuteurs). Architecture ResNet-34 allégée avec blocs Squeeze-and-Excitation, encodeur SAP (Self-Attentive Pooling) → vecteur **256-d**.

Le modèle prend la **forme d'onde brute** 16 kHz `(B, T)` et calcule en interne un spectrogramme de Mel (n_mels=40, n_fft=512, hop=160) avec InstanceNorm. Cela évite tout calcul de Mel côté Dataset et assure une cohérence exacte avec le prétraitement utilisé lors du pré-entraînement VoxCeleb.

**Poids pré-entraînés** : chargement depuis `baseline_lite_ap.model` (Oxford Robots Lab) ou tout checkpoint produit par `voxceleb_trainer`. Le loader gère les deux formats de checkpoint (nouveau `model_state_dict` avec préfixe `__S__.*`, et ancien format Oxford sans préfixe).

### Fusion & tête de classification
Concaténation des deux vecteurs (**1024 + 512 = 1536**), puis séquence **Linear(1536→`HIDDEN_DIM`) → BatchNorm → Dropout → ReLU → Linear(`HIDDEN_DIM`→`NUM_LABELS`)**. Une vérification dimensionnelle est exécutée sur le batch sonde, puis la mémoire GPU est libérée.

> ✅ **Avantage du transfer learning VoxCeleb** — ResNetSE34L a appris à encoder les caractéristiques vocales distinctives de milliers de locuteurs. Ces représentations capturent formants, harmoniques et texture de bruit laryngé — directement pertinent pour distinguer PR, LMB, fuite glottique et voix saine. Le pré-entraînement voix-spécifique est bien plus adapté à notre domaine que CNN14/AudioSet (bruits généraux) ou ResNet/ImageNet (textures visuelles).


In [12]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 6 — Modèle de fusion bi-branche (WavLM-Large + VoxCeleb ResNetSE34L)
# ═══════════════════════════════════════════════════════════════════════════
from transformers import WavLMModel
from models.ResNetSE34L import MainModel as ResNetSE34L_MainModel


# ── Chargement du backbone VoxCeleb pré-entraîné ─────────────────────────────

def load_voxceleb_backbone(ckpt_path, nOut=VOXCELEB_EMB_DIM, encoder_type='SAP'):
    """Instancie ResNetSE34L et charge les poids VoxCeleb pré-entraînés.

    Gère les deux formats de checkpoint du trainer VoxCeleb :
      • Nouveau format : {'model_state_dict': {...}} avec clés '__S__.*'
      • Ancien format  : {'model': {...}} ou dict nu (clés sans préfixe)
    """
    backbone = ResNetSE34L_MainModel(nOut=nOut, encoder_type=encoder_type)

    if not ckpt_path or not os.path.isfile(ckpt_path):
        print(f'[VoxCeleb] Checkpoint introuvable : {ckpt_path}')
        print('           → backbone initialisé aléatoirement.')
        return backbone

    print(f'[VoxCeleb] Chargement depuis : {ckpt_path}')
    raw = torch.load(ckpt_path, map_location='cpu', weights_only=False)

    if isinstance(raw, dict) and 'model_state_dict' in raw:
        state = raw['model_state_dict']          # format nouveau trainer
    elif isinstance(raw, dict) and 'model' in raw:
        state = raw['model']                     # format héritage Oxford
    else:
        state = raw                              # dict brut

    s_prefix = '__S__.'
    backbone_keys = {k for k in state
                     if k.startswith(s_prefix) or k.startswith('module.' + s_prefix)}
    bd = backbone.state_dict()
    if backbone_keys:
        clean = {}
        for k, v in state.items():
            new_k = k.replace('module.' + s_prefix, '').replace(s_prefix, '')
            if new_k in bd and bd[new_k].shape == v.shape:
                clean[new_k] = v
    else:
        # Checkpoint Oxford baseline : clés directes (pas de préfixe __S__)
        clean = {k: v for k, v in state.items()
                 if k in bd and bd[k].shape == v.shape}

    missing, unexpected = backbone.load_state_dict(clean, strict=False)
    print(f'[VoxCeleb] {len(clean)}/{len(backbone.state_dict())} tenseurs chargés '
          f'| manquants={len(missing)} | inattendus={len(unexpected)}')
    return backbone


# ── Modèle de fusion bi-branche ───────────────────────────────────────────────

class DualBranchFusion(nn.Module):
    """WavLM-Large (gelé) + ResNetSE34L VoxCeleb (affinable) → fusion 1280-d → 4 classes.

    Transfer learning : le backbone ResNetSE34L a été pré-entraîné sur VoxCeleb
    (reconnaissance du locuteur) et ses représentations vocales sont adaptées par
    fine-tuning à la détection de pathologies laryngées.
    """

    def __init__(self, wavlm_dir, num_labels, freeze_wavlm=FREEZE_WAVLM,
                 voxceleb_pretrained=VOXCELEB_PRETRAINED, dropout=DROPOUT,
                 hidden=HIDDEN_DIM, voxceleb_emb_dim=VOXCELEB_EMB_DIM):
        super().__init__()

        # ── Branche A : WavLM-Large (gelé) ───────────────────────────────────────
        self.wavlm        = WavLMModel.from_pretrained(wavlm_dir)
        self.freeze_wavlm = freeze_wavlm
        if freeze_wavlm:
            self.wavlm.eval()
            for p in self.wavlm.parameters():
                p.requires_grad = False
        wavlm_dim = self.wavlm.config.hidden_size                      # 1024

        # ── Branche B : ResNetSE34L VoxCeleb (transfer learning) ──────────────
        self.voxceleb = load_voxceleb_backbone(
            VOXCELEB_CKPT_PATH if voxceleb_pretrained else None,
            nOut=voxceleb_emb_dim,
        )

        # ── Tête de fusion ───────────────────────────────────────────────────────────
        fusion_dim = wavlm_dim + voxceleb_emb_dim                      # 1280
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.Dropout(dropout),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, num_labels),
        )
        self.fusion_dim = fusion_dim

    def train(self, mode=True):
        super().train(mode)
        if self.freeze_wavlm:
            self.wavlm.eval()
        return self

    def forward(self, input_values, wave):
        # Branche A — WavLM → (B, 1024)
        if self.freeze_wavlm:
            with torch.no_grad():
                feat_a = self.wavlm(input_values).last_hidden_state
        else:
            feat_a = self.wavlm(input_values).last_hidden_state
        feat_a = feat_a.mean(dim=1)

        # Branche B — ResNetSE34L VoxCeleb : forme d'onde brute → (B, 256)
        # Le modèle calcule en interne mel(n_mels=40) + InstanceNorm + ResNet.
        feat_b = self.voxceleb(wave)

        return self.classifier(torch.cat([feat_a, feat_b], dim=1))


def build_model():
    return DualBranchFusion(MODEL_DIR, NUM_LABELS)


# ── Vérification dimensionnelle ───────────────────────────────────────────────
_m = build_model().to(device)
_total     = sum(p.numel() for p in _m.parameters())
_trainable = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Dimension de fusion     : {_m.fusion_dim} (1024 WavLM + {VOXCELEB_EMB_DIM} VoxCeleb ResNetSE34L)')
print(f'Paramètres totaux       : {_total:,}')
print(f'Paramètres entraînables : {_trainable:,}  ({100 * _trainable / _total:.1f} %)')
print(f'Classes de sortie       : {NUM_LABELS} → {CLASSES}')

_m.eval()
with torch.no_grad():
    _logits = _m(xb.to(device), wb.to(device))
print(f'Sortie logits (test)    : {tuple(_logits.shape)}  (attendu : ({xb.shape[0]}, {NUM_LABELS}))')
assert _logits.shape == (xb.shape[0], NUM_LABELS), 'Dimension de sortie inattendue !'

del _m, _logits
gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()


Loading weights: 100%|██████████| 488/488 [00:00<00:00, 56645.27it/s]


Embedding size is 512, encoder SAP.
[VoxCeleb] Chargement depuis : C:/Users/AdminIA/Desktop/Vox_celeb/voxceleb_trainer/baseline_lite_ap.model
[VoxCeleb] 287/287 tenseurs chargés | manquants=0 | inattendus=0
Dimension de fusion     : 1536 (1024 WavLM + 512 VoxCeleb ResNetSE34L)
Paramètres totaux       : 317,680,218
Paramètres entraînables : 2,227,098  (0.7 %)
Classes de sortie       : 4 → ['PR', 'LMB', 'fuite glottique', 'Healthy']
Sortie logits (test)    : (4, 4)  (attendu : (4, 4))


## 7. Boucle de validation croisée K-Fold (entraînement + OOF)

L'entraînement, la validation et la collecte des prédictions sont entièrement encapsulés dans la **boucle sur les K plis**. Pour chaque pli :

1. **Construction des jeux** — le train reçoit les originaux du pli **plus** les segments augmentés (hors patients de validation) ; le pli de validation reste **strictement original**. Une assertion vérifie l'absence de fuite patient.
2. **Réinitialisation** — `build_model()` fournit un modèle neuf ; `AdamW` n'optimise que les paramètres entraînables (ResNetSE34L VoxCeleb + tête de fusion ; WavLM gelé).
3. **Entraînement** — AMP (`autocast` + `GradScaler`), accumulation de gradient (batch effectif 16), planificateur cosinus avec échauffement, clipping de la norme, **early stopping** sur la perte de validation (meilleur modèle sauvegardé par pli).
4. **Prédictions out-of-fold** — le meilleur modèle prédit son pli de validation ; les probabilités/labels alimentent les conteneurs OOF.
5. **Libération mémoire** — `del` + `gc.collect()` + `torch.cuda.empty_cache()` entre les plis.

Les métriques par pli sont agrégées en fin de cellule (moyenne ± écart-type).


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 7 — Boucle de validation croisée K-Fold (entraînement + OOF)
# ═══════════════════════════════════════════════════════════════════════════
from torch.amp import GradScaler, autocast
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
import torch.nn.functional as F

criterion = nn.CrossEntropyLoss()
labels_idx = np.arange(NUM_LABELS)

# Conteneurs out-of-fold : chaque enregistrement original est prédit exactement
# une fois (lorsqu'il appartient au pli de validation).
oof_labels = np.full(len(df), -1, dtype=int)
oof_preds  = np.full(len(df), -1, dtype=int)
oof_probs  = np.zeros((len(df), NUM_LABELS), dtype=float)
fold_rows, fold_curves = [], []

for fold, (tr_idx, va_idx) in enumerate(folds):
    print(f'\n{"="*70}\nPLI {fold+1}/{K_FOLDS}\n{"="*70}')

    # ── Construction des jeux du pli ─────────────────────────────────────────
    train_df = df.iloc[tr_idx].reset_index(drop=True)
    val_df   = df.iloc[va_idx].reset_index(drop=True)
    val_patients = set(val_df['Last_Name'])

    # Injection des données augmentées : UNIQUEMENT dans le train, en écartant
    # tout segment issu d'un patient présent dans le pli de validation.
    aug_fold = aug_df[~aug_df['Last_Name'].isin(val_patients)]
    train_full = pd.concat([train_df, aug_fold], ignore_index=True)

    # Garde-fou : le pli de validation reste strictement original et sans fuite.
    assert not (set(train_full['Last_Name']) & val_patients), 'Fuite patient dans le pli !'
    print(f'Train {len(train_full)} (orig {len(train_df)} + aug {len(aug_fold)}) | '
          f'Val {len(val_df)} (original uniquement)')

    train_loader = DataLoader(FochVowelDataset(train_full, processor), batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(FochVowelDataset(val_df, processor), batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=0, pin_memory=True)

    # ── Modèle / optimiseur / planificateur réinitialisés pour ce pli ────────
    model     = build_model().to(device)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = max(1, (len(train_loader) // ACCUM_STEPS)) * EPOCHS
    scheduler   = get_cosine_schedule_with_warmup(optimizer, total_steps // 10, total_steps)
    scaler      = GradScaler()

    ckpt = os.path.join(WEIGHTS_DIR, f'{RUN_NAME}_fold{fold+1}_best.pt')
    best_val_loss, patience_ctr, val_hist = float('inf'), 0, []

    for epoch in range(EPOCHS):
        # ── ENTRAÎNEMENT (précision mixte) ───────────────────────────────────
        model.train()
        ep_tr, corr, tot = 0.0, 0, 0
        optimizer.zero_grad()
        for i, (xb, wb, yb) in enumerate(train_loader):
            xb = xb.to(device, non_blocking=True)
            wb = wb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            with autocast(device_type=device.type):
                logits = model(xb, wb)
                loss   = criterion(logits, yb) / ACCUM_STEPS
            scaler.scale(loss).backward()
            ep_tr += loss.item() * ACCUM_STEPS
            corr  += (logits.detach().argmax(1) == yb).sum().item()
            tot   += yb.size(0)
            if (i + 1) % ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model.parameters()), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
        avg_tr = ep_tr / len(train_loader)

        # ── VALIDATION (pleine précision) ────────────────────────────────────
        model.eval()
        ep_v, corr_v, tot_v = 0.0, 0, 0
        with torch.no_grad():
            for xb, wb, yb in val_loader:
                xb = xb.to(device, non_blocking=True)
                wb = wb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                logits = model(xb, wb)
                loss   = criterion(logits, yb)
                ep_v   += loss.item()
                corr_v += (logits.argmax(1) == yb).sum().item()
                tot_v  += yb.size(0)
        avg_v = ep_v / len(val_loader)
        val_hist.append(avg_v)
        print(f'  Epoch {epoch+1:02d}/{EPOCHS} | Train {avg_tr:.4f}/{corr/tot:.3f} | '
              f'Val {avg_v:.4f}/{corr_v/tot_v:.3f}')

        if avg_v < best_val_loss:
            best_val_loss, patience_ctr = avg_v, 0
            torch.save(model.state_dict(), ckpt)
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f'  Arrêt anticipé à l\'epoch {epoch+1}.')
                break

    # Garde-fou : si aucune amélioration finie n'a été enregistrée, on sauvegarde
    # tout de même le dernier état pour permettre le rechargement ci-dessous.
    if not os.path.exists(ckpt):
        torch.save(model.state_dict(), ckpt)

    # ── Prédictions out-of-fold (meilleur modèle du pli, pleine précision) ───
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()
    probs_list = []
    with torch.no_grad():
        for xb, wb, yb in val_loader:
            xb = xb.to(device); wb = wb.to(device)
            probs_list.append(F.softmax(model(xb, wb), dim=1).cpu().numpy())
    probs  = np.concatenate(probs_list)                      # ordre = val_df (shuffle=False)
    preds  = probs.argmax(1)
    labels = val_df['label'].to_numpy()
    oof_probs[va_idx], oof_preds[va_idx], oof_labels[va_idx] = probs, preds, labels

    bal = balanced_accuracy_score(labels, preds)
    mf1 = f1_score(labels, preds, average='macro', zero_division=0)
    try:
        auc = roc_auc_score(labels, probs, multi_class='ovr', average='macro', labels=labels_idx)
    except ValueError:
        auc = float('nan')
    fold_rows.append({'fold': fold + 1, 'balanced_accuracy': bal, 'macro_f1': mf1,
                      'macro_auc_ovr': auc, 'n_train': len(train_full),
                      'n_aug': len(aug_fold), 'n_val': len(val_df),
                      'best_val_loss': best_val_loss})
    fold_curves.append(val_hist)
    print(f'  → Pli {fold+1} : Balanced Acc {bal:.3f} | Macro-F1 {mf1:.3f} | Macro AUC {auc:.3f}')

    # ── Libération mémoire avant le pli suivant ──────────────────────────────
    del model, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

fold_df = pd.DataFrame(fold_rows)
print(f'\n{"="*70}\nValidation croisée terminée — synthèse par pli :\n{"="*70}')
print(fold_df.to_string(index=False))
print('\nMoyenne ± écart-type sur les plis :')
for col in ['balanced_accuracy', 'macro_f1', 'macro_auc_ovr']:
    print(f'  {col:<20} : {fold_df[col].mean():.4f} ± {fold_df[col].std():.4f}')



PLI 1/5
Train 296 (orig 196 + aug 100) | Val 50 (original uniquement)


Loading weights: 100%|██████████| 488/488 [00:00<00:00, 42657.20it/s]

Embedding size is 512, encoder SAP.
[VoxCeleb] Chargement depuis : C:/Users/AdminIA/Desktop/Vox_celeb/voxceleb_trainer/baseline_lite_ap.model
[VoxCeleb] 287/287 tenseurs chargés | manquants=0 | inattendus=0


  Epoch 01/30 | Train 1.4427/0.321 | Val 1.3530/0.400
  Epoch 02/30 | Train 1.2438/0.432 | Val 1.1805/0.540
  Epoch 03/30 | Train 1.1405/0.497 | Val 1.0441/0.560
  Epoch 04/30 | Train 1.0339/0.568 | Val 0.9611/0.560
  Epoch 05/30 | Train 0.9903/0.625 | Val 0.9411/0.680
  Epoch 06/30 | Train 0.9363/0.642 | Val 0.9810/0.660
  Epoch 07/30 | Train 0.9461/0.615 | Val 0.8702/0.680
  Epoch 08/30 | Train 0.8595/0.645 | Val 0.8744/0.720
  Epoch 09/30 | Train 0.8681/0.669 | Val 0.8819/0.680
  Epoch 10/30 | Train 0.8106/0.672 | Val 0.8706/0.660
  Epoch 11/30 | Train 0.7857/0.716 | Val 0.8586/0.620
  Epoch 12/30 | Train 0.7925/0.696 | Val 0.8668/0.680
  Epoch 13/30 | Train 0.7290/0.713 | Val 0.9099/0.660
  Epoch 14/30 | Train 0.7554/0.703 | Val 0.9470/0.620
  Epoch 15/30 | Train 0.7319/0.713 | Val 0.9255/0.680
  Epoch 16/30 | Train 0.7481/0.672 | Val 0.8597/0.640
  Epoch 17/30 | Train 0.6805/0.750 | Val 0.8197/0.720
  Epoch 18/30 | Train 0.6887/0.723 | Val 0.8570/0.660
  Epoch 19/30 | Train 0.7086

Loading weights: 100%|██████████| 488/488 [00:00<00:00, 34328.80it/s]

Embedding size is 512, encoder SAP.


[VoxCeleb] Chargement depuis : C:/Users/AdminIA/Desktop/Vox_celeb/voxceleb_trainer/baseline_lite_ap.model
[VoxCeleb] 287/287 tenseurs chargés | manquants=0 | inattendus=0
  Epoch 01/30 | Train 1.4547/0.243 | Val 1.3866/0.286
  Epoch 02/30 | Train 1.3137/0.380 | Val 1.1505/0.612
  Epoch 03/30 | Train 1.0935/0.551 | Val 0.9384/0.653
  Epoch 04/30 | Train 0.9910/0.596 | Val 0.9094/0.612
  Epoch 05/30 | Train 0.9703/0.606 | Val 0.8081/0.714
  Epoch 06/30 | Train 0.9903/0.603 | Val 0.8456/0.694
  Epoch 07/30 | Train 0.8656/0.675 | Val 0.8108/0.633
  Epoch 08/30 | Train 0.8228/0.685 | Val 0.7763/0.653
  Epoch 09/30 | Train 0.8518/0.651 | Val 0.7530/0.673
  Epoch 10/30 | Train 0.7859/0.699 | Val 0.8078/0.633
  Epoch 11/30 | Train 0.6998/0.743 | Val 0.7997/0.673


## 8. Évaluation clinique agrégée (out-of-fold)

Toutes les prédictions **out-of-fold** (chaque enregistrement prédit une fois, par le modèle du pli où il sert de validation) sont rassemblées pour une évaluation non biaisée sur l'ensemble du jeu. Compte tenu du déséquilibre, on privilégie des métriques robustes :

| Métrique | Intérêt clinique |
|---|---|
| **Balanced accuracy** | Moyenne des rappels par classe — robuste au déséquilibre |
| **Macro-F1** | Traite chaque pathologie à parité |
| **Macro AUC (OvR)** | Pouvoir discriminant moyen, un-contre-tous |
| **Matrice de confusion combinée** | Confusions inter-pathologies sur tout le jeu (OOF) |
| **Métriques par pli** | Dispersion (moyenne ± écart-type) entre les plis |

Le tableau de bord agrège la matrice de confusion OOF (brute), les métriques par pli et les courbes de perte de validation de chaque pli. Tous les artefacts (PNG, JSON, CSV de confusion, CSV par pli) sont sauvegardés dans `.foch_métriques`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 8 — Évaluation agrégée out-of-fold (K-Fold)
# ═══════════════════════════════════════════════════════════════════════════
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc as sk_auc)
from sklearn.preprocessing import label_binarize

assert (oof_labels >= 0).all(), "Des enregistrements n\'ont pas été évalués !"

# ── Métriques agrégées ────────────────────────────────────────────────────────
print('Rapport de classification agrégé (out-of-fold) :')
print('\n' + classification_report(oof_labels, oof_preds, labels=labels_idx,
                                    target_names=CLASSES, digits=3, zero_division=0))
oof_bal = balanced_accuracy_score(oof_labels, oof_preds)
oof_mf1 = f1_score(oof_labels, oof_preds, average='macro', zero_division=0)
try:
    oof_auc = roc_auc_score(oof_labels, oof_probs, multi_class='ovr',
                            average='macro', labels=labels_idx)
except ValueError:
    oof_auc = float('nan')

mean_bal, std_bal = fold_df['balanced_accuracy'].mean(), fold_df['balanced_accuracy'].std()
mean_f1,  std_f1  = fold_df['macro_f1'].mean(),          fold_df['macro_f1'].std()
mean_auc, std_auc = fold_df['macro_auc_ovr'].mean(),     fold_df['macro_auc_ovr'].std()
print(f'OOF global        — Balanced Acc {oof_bal:.4f} | Macro-F1 {oof_mf1:.4f} | Macro AUC {oof_auc:.4f}')
print(f'Moyenne par pli   — Balanced Acc {mean_bal:.4f}±{std_bal:.4f} | '
      f'Macro-F1 {mean_f1:.4f}±{std_f1:.4f} | Macro AUC {mean_auc:.4f}±{std_auc:.4f}')

# ── ROC par classe (One-vs-Rest) ──────────────────────────────────────────────
oof_labels_bin = label_binarize(oof_labels, classes=labels_idx)   # (n, 4)

# ── Tableau de bord 2×2 ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f'FOCH — Fusion WavLM-Large + VoxCeleb ResNetSE34L | {K_FOLDS}-Fold CV\n'
             f'OOF Balanced Acc {oof_bal:.3f} | Macro-F1 {oof_mf1:.3f} | Macro AUC {oof_auc:.3f}',
             fontsize=14)

# ── [0, 0] Matrice de confusion (effectifs bruts) ─────────────────────────────
cm = confusion_matrix(oof_labels, oof_preds, labels=labels_idx)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
            xticklabels=CLASSES, yticklabels=CLASSES)
axes[0, 0].set_title('Matrice de confusion combinée (OOF, effectifs)')
axes[0, 0].set_ylabel('Vraie classe')
axes[0, 0].set_xlabel('Classe prédite')

# ── [0, 1] Courbes ROC OvR par classe ────────────────────────────────────────
_colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']   # rouge, bleu, vert, violet
for i, (cls, col) in enumerate(zip(CLASSES, _colors)):
    fpr, tpr, _ = roc_curve(oof_labels_bin[:, i], oof_probs[:, i])
    auc_i = sk_auc(fpr, tpr)
    axes[0, 1].plot(fpr, tpr, color=col, lw=2, label=f'{cls}  (AUC = {auc_i:.3f})')

axes[0, 1].plot([0, 1], [0, 1], color='grey', lw=1, linestyle='--', label='Aléatoire')
axes[0, 1].fill_between([0, 1], [0, 1], alpha=0.04, color='grey')
axes[0, 1].set_xlim([-0.01, 1.01])
axes[0, 1].set_ylim([-0.01, 1.02])
axes[0, 1].set_xlabel('Taux de faux positifs (1 − Spécificité)')
axes[0, 1].set_ylabel('Taux de vrais positifs (Sensibilité)')
axes[0, 1].set_title(f'Courbes ROC One-vs-Rest (Macro AUC = {oof_auc:.3f})')
axes[0, 1].legend(loc='lower right', framealpha=0.9)
axes[0, 1].grid(True, alpha=0.25)

# ── [1, 0] Métriques par pli ──────────────────────────────────────────────────
x = np.arange(K_FOLDS); w = 0.27
axes[1, 0].bar(x - w, fold_df['balanced_accuracy'], w, label='Balanced Acc')
axes[1, 0].bar(x,     fold_df['macro_f1'],          w, label='Macro-F1')
axes[1, 0].bar(x + w, fold_df['macro_auc_ovr'],     w, label='Macro AUC')
axes[1, 0].set_title('Métriques par pli')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels([f'Pli {i+1}' for i in range(K_FOLDS)])
axes[1, 0].set_ylim(0, 1)
axes[1, 0].legend()
axes[1, 0].grid(True, axis='y', alpha=0.3)

# ── [1, 1] Perte de validation par pli ────────────────────────────────────────
for k, hist in enumerate(fold_curves):
    axes[1, 1].plot(range(1, len(hist) + 1), hist, 'o-', label=f'Pli {k+1}', markersize=3)
axes[1, 1].set_title('Perte de validation par pli')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Perte (entropie croisée)')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.subplots_adjust(top=0.90)
fig_path = os.path.join(METRICS_DIR, f'{RUN_NAME}_dashboard.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

# ── Sauvegarde des artefacts ──────────────────────────────────────────────────
report_dict = classification_report(oof_labels, oof_preds, labels=labels_idx,
                                     target_names=CLASSES, output_dict=True, zero_division=0)
metrics = {
    'run_name':               RUN_NAME,
    'architecture':           f'DualBranchFusion (WavLM-Large + VoxCeleb ResNetSE34L, emb_dim={VOXCELEB_EMB_DIM})',
    'cv':                     f'StratifiedKFold(n_splits={K_FOLDS})',
    'oof_balanced_accuracy':  oof_bal,
    'oof_macro_f1':           oof_mf1,
    'oof_macro_auc_ovr':      oof_auc,
    'mean_balanced_accuracy': mean_bal, 'std_balanced_accuracy': std_bal,
    'mean_macro_f1':          mean_f1,  'std_macro_f1':          std_f1,
    'mean_macro_auc_ovr':     mean_auc, 'std_macro_auc_ovr':     std_auc,
    'per_fold':               fold_rows,
    'per_class_oof':          report_dict,
    'classes':                CLASSES,
    'n_total':                int(len(df)),
}
with open(os.path.join(METRICS_DIR, f'{RUN_NAME}_metrics.json'), 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2, default=float)
np.savetxt(os.path.join(METRICS_DIR, f'{RUN_NAME}_confusion.csv'), cm, fmt='%d', delimiter=',')
fold_df.to_csv(os.path.join(METRICS_DIR, f'{RUN_NAME}_fold_metrics.csv'), index=False)

print(f'\nArtefacts sauvegardés dans {METRICS_DIR} :')
print(f'  • {RUN_NAME}_dashboard.png')
print(f'  • {RUN_NAME}_metrics.json')
print(f'  • {RUN_NAME}_confusion.csv')
print(f'  • {RUN_NAME}_fold_metrics.csv')


## 9. Synthèse et prochaines étapes

Ce notebook met en œuvre un **modèle de fusion bi-branche** (WavLM-Large + ResNetSE34L VoxCeleb) évalué par **validation croisée stratifiée à 5 plis**, avec **augmentation de données** injectée pli par pli.

**Choix méthodologiques clés :**
- **Transfer learning VoxCeleb** : ResNetSE34L pré-entraîné sur la reconnaissance du locuteur (VoxCeleb,   milliers de locuteurs, millions d'utterances) fine-tuné pour la détection de pathologies laryngées.
- **Fusion de deux vues complémentaires** : embeddings WavLM 1024-d (microscopique, court-terme)   ⊕ ResNetSE34L 512-d (voix-spécifique, VoxCeleb) → 1536-d.
- **Entrée forme d'onde brute** : ResNetSE34L prend directement `wave (B, T)` et calcule son propre   mel (n_mels=40) + InstanceNorm en interne — cohérence garantie avec le pré-entraînement VoxCeleb.
- **Chargement robuste** : le loader gère les deux formats de checkpoint VoxCeleb   (nouveau `model_state_dict` + préfixe `__S__`, ancien Oxford sans préfixe).
- **Validation croisée K-Fold** : métriques OOF + moyenne ± écart-type par pli.
- **Augmentation sans fuite** : segments injectés uniquement dans le train, patients de validation exclus.

**Pistes d'amélioration ultérieures :**
1. **Dégel progressif** de ResNetSE34L (*layer-wise learning rate decay*) — commencer gelé, débloquer    les couches supérieures après quelques epochs.
2. **Dégel progressif** de WavLM.
3. **SpecAugment** sur l'entrée WavLM.
4. **Ensembling** des K modèles de plis à l'inférence.
5. Tester **ResNetSE34V2** (ASP encoder, 512-d) pour une capacité de représentation accrue.
6. **Audit qualité** des doublons d'homonymes (`MARTIN`) via `patient_id`.

> ⚠️ **Rappel d'exécution** — cellules à exécuter **dans l'ordre** (1 → 8). La Cellule 1 ajoute `voxceleb_trainer/` au `sys.path` pour permettre l'import de `ResNetSE34L`. La Cellule 6 charge le checkpoint VoxCeleb depuis `VOXCELEB_CKPT_PATH` (télécharger `baseline_lite_ap.model` si absent). La Cellule 7 entraîne **K = 5 modèles** (WavLM gelé + ResNetSE34L + tête) : prévoir un GPU CUDA.
